In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/Rainfall.csv', encoding='latin1')

In [7]:
df = df.reset_index(drop=True)

n = len(df)

df["rain_prob"] = np.concatenate([
    np.random.uniform(0,0.3, n//3),
    np.random.uniform(0.3,0.6, n//3),
    np.random.uniform(0.6,1, n - 2*(n//3))
])

In [8]:
orders = []
delivery_time = []
delays = []
cancellations = []

for prob in df["rain_prob"]:
    
    base_orders = 100
    
    if prob > 0.7:
        orders.append(base_orders + np.random.randint(20,50))
        delivery_time.append(np.random.randint(40,60))
        delays.append(np.random.randint(20,40))
        cancellations.append(np.random.randint(10,25))
        
    elif prob > 0.4:
        orders.append(base_orders + np.random.randint(0,25))
        delivery_time.append(np.random.randint(30,45))
        delays.append(np.random.randint(10,25))
        cancellations.append(np.random.randint(5,15))
        
    else:
        orders.append(base_orders - np.random.randint(0,20))
        delivery_time.append(np.random.randint(20,35))
        delays.append(np.random.randint(5,15))
        cancellations.append(np.random.randint(2,8))

df["orders"] = orders
df["delivery_time"] = delivery_time
df["delays"] = delays
df["cancellations"] = cancellations

In [9]:
df["date"] = pd.date_range(start="2023-01-01", periods=len(df))
df["day"] = df["date"].dt.day_name()

df.loc[df["day"].isin(["Saturday","Sunday"]), "orders"] += 20

In [10]:
df["delay_pct"] = df["delays"] / df["orders"]
df["cancel_pct"] = df["cancellations"] / df["orders"]

In [11]:
df.groupby(pd.cut(df["rain_prob"], bins=3))[["orders","delivery_time","delay_pct"]].mean()

C:\Users\sujal\AppData\Local\Temp\ipykernel_22508\4167366395.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df["rain_prob"], bins=3))[["orders","delivery_time","delay_pct"]].mean()


,orders,delivery_time,delay_pct
rain_prob,,,
"(0.0019, 0.335]",97.635036,26.751825,0.099110
"(0.335, 0.667]",113.196850,35.401575,0.132723
"(0.667, 0.999]",138.127451,48.862745,0.209666


In [12]:
df.to_csv("delivery_data.csv", index=False)

In [13]:
def recommend_action(prob):
    if prob > 0.7:
        return "Increase fleet by 20%, adjust ETAs, pre-stock inventory"
    elif prob > 0.4:
        return "Monitor demand, add slight buffer to delivery time"
    else:
        return "Normal operations"

df["recommendation"] = df["rain_prob"].apply(recommend_action)